# Fund strategy classification by entity name

One method, applied uniformly: classify each of the 152 hedge funds by its **legal entity name**,
with missing names filled from the public GLEIF registry. Keyword rules map names to strategies
("… Fixed Income …", "… Relative Value …", "… Macro …"); funds whose name carries no strategy
keyword stay **Unclassified** — no secondary sources, no patchwork.

Outputs: `fund_classification/fund_strategies_by_name.csv` (one row per fund) and
`summary_by_name.csv` + a LaTeX tabular.

In [ ]:
import json, re, time
from pathlib import Path

import pandas as pd
import requests

ROOT = Path.cwd() if (Path.cwd() / "hf_Valeri.xlsx").exists() else Path.cwd().parent
OUT = ROOT / "fund_classification"
(OUT / "cache").mkdir(parents=True, exist_ok=True)

funds = pd.read_excel(ROOT / "hf_Valeri.xlsx").rename(columns={"entity_id": "lei"})
funds["lei"] = funds["lei"].str.strip().str.upper()
print(len(funds), "funds,", funds["name"].isna().sum(), "without a name in the extract")

## 1. Fill missing names from GLEIF (cached; ~2 min on first run)

In [ ]:
def gleif_name(lei):
    f = OUT / "cache" / f"{lei}.json"
    if not f.exists():
        r = requests.get(f"https://api.gleif.org/api/v1/lei-records/{lei}", timeout=30)
        time.sleep(1.05)                       # GLEIF rate limit ~1 request/sec
        if not r.ok:
            return None
        f.write_text(r.text)
    ent = json.loads(f.read_text()).get("data", {}).get("attributes", {}).get("entity", {})
    return (ent.get("legalName") or {}).get("name")

funds["name_source"] = funds["name"].notna().map({True: "SFTDS extract", False: "GLEIF"})
funds["name"] = funds["name"].fillna(funds["lei"].map(gleif_name))
print("still unnamed:", funds["name"].isna().sum())

## 2. Classify by name

First matching rule wins, top to bottom. Edit the keyword lists to taste — they are the entire
method, so the paper can state them verbatim in a footnote.

In [ ]:
RULES = [
    ("Fixed income / rates RV", ["FIXED INCOME", "FIRV", "RELATIVE VALUE", " RATES", "G-10",
        "GLOBAL RATES", "INFLATION", "BOND", "TERM CREDIT", "CONVEX", "TAIL RISK", "VOLATILITY"]),
    ("Global macro", ["MACRO", "ALL WEATHER", "PURE ALPHA", "OPTIMAL PORTFOLIO", "DMO"]),
    ("Credit", ["CREDIT", "ABS ", "HIGH YIELD", "DISTRESSED"]),
    ("Equity", ["EQUITY"]),
    ("Commodity", ["COMMODITY"]),
    ("Multi-strategy platform", ["MULTI-STRATEGY", "MULTI STRATEGY", "DIVERSIFIED ALPHA"]),
]

def classify(text):
    if not isinstance(text, str):
        return "Unclassified"
    t = " " + re.sub(r"\s+", " ", re.sub(r"[^A-Z0-9\- ]", " ", text.upper())) + " "
    return next((lab for lab, kws in RULES if any(k in t for k in kws)), "Unclassified")

funds["strategy_from_name"] = funds["name"].map(classify)
print(funds["strategy_from_name"].value_counts().to_string())

## 3. Outputs

In [ ]:
funds[["lei", "name", "name_source", "strategy_from_name"]].to_csv(
    OUT / "fund_strategies_by_name.csv", index=False)

summary = (funds["strategy_from_name"].value_counts()
           .rename_axis("strategy").rename("n_funds").reset_index())
summary["share_pct"] = (100 * summary["n_funds"] / len(funds)).round(1)
summary.to_csv(OUT / "summary_by_name.csv", index=False)
print(summary.to_string(index=False), "\n")

for _, r in summary.iterrows():                    # LaTeX rows
    print(f"{r['strategy']} & {r['n_funds']} & {r['share_pct']} \\\\")

print("\nunclassified funds:")
for n in sorted(funds.loc[funds["strategy_from_name"] == "Unclassified", "name"].dropna()):
    print(" ", n)

**Volume weighting (ECB-side):** merge `fund_strategies_by_name.csv` with the internal per-fund
volumes (`hf.xlsx` from `build_main_panel.ipynb`) on `lei` = `entity_id` and aggregate volume by
strategy. Names are also informative about volume: check whether the classified 53% of funds cover a
larger share of repo volume — the big FI-RV and macro vehicles are mostly in the classified group.